# EduVerse AI: Smart College Recommendation Engine for Tamil Nadu
## Machine Learning & Content-Based Recommendation Research Notebook
**Project**: MCA Final Year Project — EduVerse AI  
**Scope**: Higher Education Institutions in Tamil Nadu, India  
**Methodology**: Multi-Criteria Content-Based Filtering (CBF) & Vector Space Similarity

## 1. Problem Definition

Students navigating tertiary education in Tamil Nadu face major systemic obstacles:
1. **Scattered Information**: Seat intake, fee schedules, and infrastructure data are fragmented across multiple university and government portals.
2. **Opaque Multi-Attribute Trade-offs**: Choosing a college requires balancing conflicting constraints (e.g. course availability, geographic proximity, annual tuition ceiling, institutional governance type, and campus amenities).
3. **Cold-Start & Lack of Labeled Interaction Data**: In higher education admissions, users apply once every several years. There is no continuous stream of user click-logs or implicit ratings to support collaborative filtering.

**Objective**:
Develop an explainable, content-based recommendation engine that maps a student preference vector $\vec{u}$ into a ranked list of candidate institutions $C$ using normalized multi-criteria utility scoring and vector similarity.

## 2. Dataset Loading

We load institutional records directly from the application database or structured catalog.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

# Connect to application environment
sys.path.insert(0, os.path.abspath('../../'))
from app import create_app, db
from app.models import College, Course, Facility

app = create_app('development')
with app.app_context():
    colleges = College.query.all()
    print(f"Loaded {len(colleges)} institutions from database.")

## 3. Data Inspection

Inspect the schema, attribute distributions, and data types.

In [ ]:
with app.app_context():
    data = []
    for c in colleges:
        courses = [crs.course_name for crs in c.courses]
        fees = [crs.annual_fee for crs in c.courses if crs.annual_fee is not None]
        data.append({
            'college_id': c.id,
            'name': c.name,
            'district': c.district,
            'city': c.city,
            'college_type': c.college_type,
            'courses': courses,
            'facilities': [f.facility_name for f in c.facilities],
            'min_fee': min(fees) if fees else None,
            'max_fee': max(fees) if fees else None,
            'data_status': c.data_status,
            'has_coordinates': c.has_valid_coordinates
        })
    df = pd.DataFrame(data)

df.head()

## 4. Data Cleaning & Integrity Auditing

We standardize district names, trim extraneous whitespace, and ensure categorical values follow standardized nomenclatures.

In [ ]:
df['district'] = df['district'].str.strip().str.title()
df['city'] = df['city'].str.strip().str.title()
df['college_type'] = df['college_type'].str.strip()
print("Unique Districts:", df['district'].unique())
print("Unique Institution Types:", df['college_type'].unique())

## 5. Missing-Value Analysis

**Ethical Principle**: Never coerce missing data into `0` (which would falsely imply a free college) or impute arbitrary mean values into official fees. Missing values must be recognized and handled with transparent confidence penalties.

In [ ]:
missing_counts = df.isnull().sum()
print("Missing Value Summary:")
print(missing_counts)

# Check status distribution
print("\nData Status Transparency Split:")
print(df['data_status'].value_counts())

## 6. Feature Engineering

We extract three distinct feature sets:
1. **Categorical Features**: District, City, College Type.
2. **Multi-Label Categorical Sets**: Course offerings, Campus facilities.
3. **Numerical Attributes**: Scaled annual tuition fee bounds.

In [ ]:
# Multi-label binarization for campus facilities
mlb_facilities = MultiLabelBinarizer()
facilities_encoded = mlb_facilities.fit_transform(df['facilities'])
facilities_df = pd.DataFrame(facilities_encoded, columns=mlb_facilities.classes_, index=df.index)

# Multi-label binarization for degree courses
mlb_courses = MultiLabelBinarizer()
courses_encoded = mlb_courses.fit_transform(df['courses'])
courses_df = pd.DataFrame(courses_encoded, columns=[f"crs_{c}" for c in mlb_courses.classes_], index=df.index)

print(f"Facilities vocabulary ({len(mlb_facilities.classes_)}):", mlb_facilities.classes_[:5])
print(f"Courses vocabulary ({len(mlb_courses.classes_)}):", mlb_courses.classes_[:5])

## 7. Categorical Encoding with Scikit-Learn

We use `OneHotEncoder` to vectorize demographic and governance variables.

In [ ]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_features = ohe.fit_transform(df[['district', 'college_type']])
cat_feature_names = ohe.get_feature_names_out(['district', 'college_type'])
cat_df = pd.DataFrame(cat_features, columns=cat_feature_names, index=df.index)

# Concatenate into comprehensive institution feature matrix
feature_matrix = pd.concat([cat_df, facilities_df, courses_df], axis=1)
print("Total Vector Dimensions:", feature_matrix.shape[1])

## 8. Multi-Criteria Scoring & Similarity Method

Rather than treating college recommendation as a black box, we formulate an explicit, explainable utility function:

$$\text{Score} = w_1 S_{\text{course}} + w_2 S_{\text{location}} + w_3 S_{\text{budget}} + w_4 S_{\text{type}} + w_5 S_{\text{facilities}}$$

Where weights are configured as:
- $w_1 = 0.35$ (Course compatibility)
- $w_2 = 0.25$ (Geographic affinity)
- $w_3 = 0.20$ (Budget tolerance)
- $w_4 = 0.10$ (College governance type)
- $w_5 = 0.10$ (Facility coverage)

Sum of weights: $0.35 + 0.25 + 0.20 + 0.10 + 0.10 = 1.00$ (100%).

In [ ]:
from app.recommender.scoring import RecommendationScorer

scorer = RecommendationScorer()
print("Active Model Weights:", scorer.weights)

## 9. Example Recommendation Simulations

Simulate realistic student queries and inspect the generated recommendations and diagnostic factors.

In [ ]:
from app.recommender.recommender import ContentBasedRecommender

recommender = ContentBasedRecommender()

# Query 1: Student seeking MCA in Chennai within ₹1,00,000 budget
sample_preference = {
    'course': 'MCA',
    'district': 'Chennai',
    'city': 'Chennai',
    'budget': 100000,
    'college_type': '',
    'facilities': ['Hostel', 'Placement Cell', 'Central Library']
}

with app.app_context():
    results = recommender.recommend(colleges, sample_preference, top_k=5)
    for i, r in enumerate(results):
        print(f"#{i+1}: {r['college_name']} | Score: {r['score']}% | Fee: ₹{r['fee']}")
        for f in r['matching_factors']:
            print(f"   {f}")
        print("-" * 60)

## 10. Evaluation Methodology

### Why Supervised Learning is Inapplicable Here
In educational recommendation, true supervisory ground-truth labels (such as conversion, enrollment, or 5-star ratings) do not exist at academic project inception. Training a neural network or logistic classifier on synthetically manufactured target labels would constitute scientific fraud ("fake accuracy").

### Validation Criteria for Content-Based Recommenders
1. **Constraint Satisfaction Rate**: Colleges returned for course $X$ must reliably offer course $X$.
2. **Monotonic Cost Penalty**: An institution with tuition higher than the student budget ceiling must receive a lower budget subscore than an identical institution under budget.
3. **Geographic Localization Fidelity**: Colleges within the target district must score strictly higher on location affinity than colleges outside the target district.
4. **Explainability & Transparency**: 100% of recommendations must be backed by reproducible factor explanations.

In [ ]:
# Unit validation of scoring monotonicity
with app.app_context():
    res_chennai = recommender.recommend(colleges, {'course': 'MCA', 'district': 'Chennai', 'budget': 50000}, top_k=10)
    res_coimbatore = recommender.recommend(colleges, {'course': 'MCA', 'district': 'Coimbatore', 'budget': 50000}, top_k=10)

    assert len(res_chennai) > 0, "No colleges found for Chennai MCA"
    assert res_chennai[0]['district'] == 'Chennai', "Top result should be in Chennai"
    assert res_coimbatore[0]['district'] == 'Coimbatore', "Top result should be in Coimbatore"
    print("Validation Passed: District affinity respects target locality.")

## 11. System Limitations

1. **Cold Data Refresh**: Institutional fees and quota structures are subject to annual government notifications (TNEA / TANCET / DOTE) and require periodic administrative audits.
2. **Subjective Campus Culture**: Quantitative scoring models cannot measure subjective variables like campus atmosphere, extracurricular life, or student peer interactions.
3. **Decision-Support Scope**: Recommendations provide directional guidance; students must confirm seat allocation with the Directorate of Technical Education.

## 12. Export & Application Integration

The tested `RecommendationScorer` and `ContentBasedRecommender` are exported directly into the production Flask application under `app.recommender` and exposed via the REST API at `POST /api/recommend`.